In [2]:
import duckdb
import pandas as pd

# Configurações pra exibir tabelas bonitas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Conecta ao DuckDB (cria arquivo olist.db na pasta)
con = duckdb.connect('olist.db')

print("✅ Setup pronto!")

✅ Setup pronto!


In [3]:
con.execute("""
CREATE OR REPLACE TABLE orders AS 
    SELECT * FROM read_csv_auto('data/raw/olist_orders_dataset.csv');

CREATE OR REPLACE TABLE customers AS 
    SELECT * FROM read_csv_auto('data/raw/olist_customers_dataset.csv');

CREATE OR REPLACE TABLE order_items AS 
    SELECT * FROM read_csv_auto('data/raw/olist_order_items_dataset.csv');

CREATE OR REPLACE TABLE products AS 
    SELECT * FROM read_csv_auto('data/raw/olist_products_dataset.csv');

CREATE OR REPLACE TABLE sellers AS 
    SELECT * FROM read_csv_auto('data/raw/olist_sellers_dataset.csv');

CREATE OR REPLACE TABLE order_payments AS 
    SELECT * FROM read_csv_auto('data/raw/olist_order_payments_dataset.csv');

CREATE OR REPLACE TABLE order_reviews AS 
    SELECT * FROM read_csv_auto('data/raw/olist_order_reviews_dataset.csv');

CREATE OR REPLACE TABLE category_translation AS 
    SELECT * FROM read_csv_auto('data/raw/product_category_name_translation.csv');
""")

print("✅ 8 tabelas carregadas!")

✅ 8 tabelas carregadas!


In [4]:
q1 = con.execute("""
SELECT 
    DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS total_revenue,
    ROUND(AVG(oi.price + oi.freight_value), 2) AS avg_ticket
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY 1
""").df()

# === INSIGHTS ===
print("=" * 60)
print("📊 Q1 — RECEITA MENSAL")
print("=" * 60)
print(f"📅 Período analisado: {q1['month'].min().strftime('%b/%Y')} até {q1['month'].max().strftime('%b/%Y')}")
print(f"💰 Receita total: R$ {q1['total_revenue'].sum():,.2f}")
print(f"📦 Pedidos totais: {q1['total_orders'].sum():,}")
print(f"🎫 Ticket médio geral: R$ {q1['total_revenue'].sum()/q1['total_orders'].sum():,.2f}")
melhor = q1.loc[q1['total_revenue'].idxmax()]
print(f"🏆 Melhor mês: {melhor['month'].strftime('%b/%Y')} com R$ {melhor['total_revenue']:,.2f}")
print("=" * 60)

q1

📊 Q1 — RECEITA MENSAL
📅 Período analisado: Sep/2016 até Aug/2018
💰 Receita total: R$ 15,419,773.75
📦 Pedidos totais: 96,478
🎫 Ticket médio geral: R$ 159.83
🏆 Melhor mês: Nov/2017 com R$ 1,153,364.20


,month,total_orders,total_revenue,avg_ticket
0,2016-09-01,1,143.46,47.82
1,2016-10-01,265,"46,490.66",148.53
2,2016-12-01,1,19.62,19.62
3,2017-01-01,750,"127,482.37",139.63
4,2017-02-01,1653,"271,239.32",145.98
5,2017-03-01,2546,"414,330.95",143.02
6,2017-04-01,2303,"390,812.40",152.13
7,2017-05-01,3546,"566,851.40",141.57
8,2017-06-01,3135,"490,050.37",140.46
9,2017-07-01,3872,"566,299.08",128.24


In [5]:
q2 = con.execute("""
SELECT 
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS total_revenue,
    ROUND(AVG(oi.price + oi.freight_value), 2) AS avg_ticket,
    ROUND(100.0 * SUM(oi.price + oi.freight_value) 
          / SUM(SUM(oi.price + oi.freight_value)) OVER (), 2) AS pct_of_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY total_revenue DESC
""").df()

print("=" * 60)
print("📊 Q2 — RECEITA POR ESTADO")
print("=" * 60)
top3 = q2.head(3)
print(f"🥇 Top 3 estados concentram {top3['pct_of_revenue'].sum():.1f}% da receita:")
for _, row in top3.iterrows():
    print(f"   {row['customer_state']}: R$ {row['total_revenue']:,.2f} ({row['pct_of_revenue']}%)")
print(f"\n🎫 Estado com maior ticket médio: {q2.loc[q2['avg_ticket'].idxmax(), 'customer_state']} (R$ {q2['avg_ticket'].max():,.2f})")
print(f"🎫 Estado com menor ticket médio: {q2.loc[q2['avg_ticket'].idxmin(), 'customer_state']} (R$ {q2['avg_ticket'].min():,.2f})")
print("=" * 60)

q2

📊 Q2 — RECEITA POR ESTADO
🥇 Top 3 estados concentram 62.5% da receita:
   SP: R$ 5,769,703.15 (37.42%)
   RJ: R$ 2,055,401.57 (13.33%)
   MG: R$ 1,818,891.67 (11.8%)

🎫 Estado com maior ticket médio: PB (R$ 235.22)
🎫 Estado com menor ticket médio: SP (R$ 124.22)


,customer_state,total_orders,total_revenue,avg_ticket,pct_of_revenue
0,SP,40501,"5,769,703.15",124.22,37.42
1,RJ,12350,"2,055,401.57",145.33,13.33
2,MG,11354,"1,818,891.67",140.82,11.80
3,RS,5345,"861,472.79",140.44,5.59
4,PR,4923,"781,708.80",138.38,5.07
5,SC,3546,"595,127.78",145.26,3.86
6,BA,3256,"591,137.81",160.50,3.83
7,DF,2080,"346,123.35",146.97,2.24
8,GO,1957,"334,212.35",146.78,2.17
9,ES,1995,"317,657.93",142.77,2.06


In [6]:
q3 = con.execute("""
WITH monthly_revenue AS (
    SELECT 
        DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
        SUM(oi.price + oi.freight_value) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY 1
)
SELECT 
    month,
    ROUND(revenue, 2) AS revenue,
    ROUND(LAG(revenue) OVER (ORDER BY month), 2) AS previous_month,
    ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month)) 
          / LAG(revenue) OVER (ORDER BY month), 2) AS mom_growth_pct
FROM monthly_revenue
ORDER BY month
""").df()

print("=" * 60)
print("📊 Q3 — CRESCIMENTO MÊS A MÊS (MoM)")
print("=" * 60)
q3_clean = q3.dropna(subset=['mom_growth_pct'])
print(f"📈 Crescimento MoM médio: {q3_clean['mom_growth_pct'].mean():.2f}%")
melhor = q3_clean.loc[q3_clean['mom_growth_pct'].idxmax()]
pior = q3_clean.loc[q3_clean['mom_growth_pct'].idxmin()]
print(f"🚀 Melhor mês de crescimento: {melhor['month'].strftime('%b/%Y')} (+{melhor['mom_growth_pct']}%)")
print(f"📉 Pior mês: {pior['month'].strftime('%b/%Y')} ({pior['mom_growth_pct']}%)")
print("=" * 60)

q3

📊 Q3 — CRESCIMENTO MÊS A MÊS (MoM)
📈 Crescimento MoM médio: 31006.75%
🚀 Melhor mês de crescimento: Jan/2017 (+649657.24%)
📉 Pior mês: Dec/2016 (-99.96%)


,month,revenue,previous_month,mom_growth_pct
0,2016-09-01,143.46,NaN,NaN
1,2016-10-01,"46,490.66",143.46,"32,306.71"
2,2016-12-01,19.62,"46,490.66",-99.96
3,2017-01-01,"127,482.37",19.62,"649,657.24"
4,2017-02-01,"271,239.32","127,482.37",112.77
5,2017-03-01,"414,330.95","271,239.32",52.75
6,2017-04-01,"390,812.40","414,330.95",-5.68
7,2017-05-01,"566,851.40","390,812.40",45.04
8,2017-06-01,"490,050.37","566,851.40",-13.55
9,2017-07-01,"566,299.08","490,050.37",15.56


In [7]:
q4 = con.execute("""
WITH customer_orders AS (
    SELECT 
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY 1
)
SELECT 
    CASE 
        WHEN total_orders = 1 THEN '1 - One-time buyer'
        WHEN total_orders BETWEEN 2 AND 3 THEN '2 - Low recurring'
        ELSE '3 - High recurring (4+)'
    END AS customer_segment,
    COUNT(*) AS customer_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_base
FROM customer_orders
GROUP BY 1
ORDER BY 1
""").df()

print("=" * 60)
print("📊 Q4 — CLIENTES RECORRENTES VS ÚNICOS")
print("=" * 60)
total = q4['customer_count'].sum()
unicos = q4[q4['customer_segment'].str.contains('One-time')]['customer_count'].values[0]
recorrentes = total - unicos
print(f"👥 Total de clientes: {total:,}")
print(f"🔹 Únicos (1 compra): {unicos:,} ({100*unicos/total:.1f}%)")
print(f"🔁 Recorrentes (2+ compras): {recorrentes:,} ({100*recorrentes/total:.1f}%)")
print(f"\n💡 INSIGHT: Apenas {100*recorrentes/total:.1f}% retornam — ENORME oportunidade de retenção")
print("=" * 60)

q4

📊 Q4 — CLIENTES RECORRENTES VS ÚNICOS
👥 Total de clientes: 93,358
🔹 Únicos (1 compra): 90,557 (97.0%)
🔁 Recorrentes (2+ compras): 2,801 (3.0%)

💡 INSIGHT: Apenas 3.0% retornam — ENORME oportunidade de retenção


,customer_segment,customer_count,pct_of_base
0,1 - One-time buyer,90557,97.00
1,2 - Low recurring,2754,2.95
2,3 - High recurring (4+),47,0.05


In [8]:
q5 = con.execute("""
SELECT 
    c.customer_city,
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS total_revenue,
    ROUND(AVG(oi.price + oi.freight_value), 2) AS avg_ticket
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1, 2
ORDER BY total_revenue DESC
LIMIT 10
""").df()

print("=" * 60)
print("📊 Q5 — TOP 10 CIDADES POR RECEITA")
print("=" * 60)
total_top10 = q5['total_revenue'].sum()
print(f"🏙️  Top 10 cidades = R$ {total_top10:,.2f} em receita")
print(f"\n🥇 Top 3:")
for i, row in q5.head(3).iterrows():
    print(f"   {row['customer_city'].title()}/{row['customer_state']}: R$ {row['total_revenue']:,.2f}")
print("=" * 60)

q5

📊 Q5 — TOP 10 CIDADES POR RECEITA
🏙️  Top 10 cidades = R$ 5,134,007.31 em receita

🥇 Top 3:
   Sao Paulo/SP: R$ 2,107,960.17
   Rio De Janeiro/RJ: R$ 1,111,732.21
   Belo Horizonte/MG: R$ 405,950.51


,customer_city,customer_state,total_orders,total_revenue,avg_ticket
0,sao paulo,SP,15045,"2,107,960.17",121.15
1,rio de janeiro,RJ,6601,"1,111,732.21",146.43
2,belo horizonte,MG,2697,"405,950.51",131.50
3,brasilia,DF,2071,"345,199.05",147.46
4,curitiba,PR,1489,"238,459.72",138.08
5,porto alegre,RS,1342,"214,805.84",136.56
6,campinas,SP,1406,"209,002.90",128.54
7,salvador,BA,1188,"207,713.30",152.96
8,guarulhos,SP,1144,"157,735.65",121.90
9,niteroi,RJ,825,"135,447.96",141.39


In [9]:
q6 = con.execute("""
WITH customer_purchases AS (
    SELECT 
        c.customer_unique_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id 
            ORDER BY o.order_purchase_timestamp
        ) AS purchase_number
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
),
first_and_second AS (
    SELECT 
        customer_unique_id,
        MAX(CASE WHEN purchase_number = 1 THEN order_purchase_timestamp END) AS first_purchase,
        MAX(CASE WHEN purchase_number = 2 THEN order_purchase_timestamp END) AS second_purchase
    FROM customer_purchases
    WHERE purchase_number <= 2
    GROUP BY 1
    HAVING COUNT(*) = 2
)
SELECT 
    COUNT(*) AS customers_with_repeat,
    ROUND(AVG(DATE_DIFF('day', first_purchase, second_purchase)), 1) AS avg_days_to_repeat,
    ROUND(MEDIAN(DATE_DIFF('day', first_purchase, second_purchase)), 1) AS median_days_to_repeat
FROM first_and_second
""").df()

print("=" * 60)
print("📊 Q6 — TEMPO ENTRE 1ª E 2ª COMPRA")
print("=" * 60)
row = q6.iloc[0]
print(f"🔁 Clientes que voltaram: {int(row['customers_with_repeat']):,}")
print(f"📅 Tempo médio entre compras: {row['avg_days_to_repeat']} dias")
print(f"📅 Mediana: {row['median_days_to_repeat']} dias")
print(f"\n💡 INSIGHT: Janela ideal de remarketing = ~{int(row['median_days_to_repeat'])} dias após a primeira compra")
print("=" * 60)

q6

📊 Q6 — TEMPO ENTRE 1ª E 2ª COMPRA
🔁 Clientes que voltaram: 2,801
📅 Tempo médio entre compras: 81.2 dias
📅 Mediana: 29.0 dias

💡 INSIGHT: Janela ideal de remarketing = ~29 dias após a primeira compra


,customers_with_repeat,avg_days_to_repeat,median_days_to_repeat
0,2801,81.20,29.00


In [10]:
q7 = con.execute("""
SELECT 
    COALESCE(ct.product_category_name_english, p.product_category_name) AS category,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(*) AS total_items_sold,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(AVG(oi.price), 2) AS avg_price,
    RANK() OVER (ORDER BY SUM(oi.price) DESC) AS rank_by_revenue,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS rank_by_volume
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN category_translation ct 
    ON p.product_category_name = ct.product_category_name
JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered' 
    AND p.product_category_name IS NOT NULL
GROUP BY 1
ORDER BY total_revenue DESC
LIMIT 10
""").df()

print("=" * 60)
print("📊 Q7 — TOP 10 CATEGORIAS")
print("=" * 60)
top_receita = q7.iloc[0]
top_volume = q7.sort_values('total_items_sold', ascending=False).iloc[0]
print(f"💰 Líder em RECEITA: {top_receita['category']} (R$ {top_receita['total_revenue']:,.2f})")
print(f"📦 Líder em VOLUME: {top_volume['category']} ({int(top_volume['total_items_sold']):,} itens)")
if top_receita['category'] != top_volume['category']:
    print(f"\n💡 INSIGHT: Líder em receita ≠ líder em volume — estratégias diferentes pra cada uma!")
print("=" * 60)

q7

📊 Q7 — TOP 10 CATEGORIAS
💰 Líder em RECEITA: health_beauty (R$ 1,233,131.72)
📦 Líder em VOLUME: bed_bath_table (10,953 itens)

💡 INSIGHT: Líder em receita ≠ líder em volume — estratégias diferentes pra cada uma!


,category,total_orders,total_items_sold,total_revenue,avg_price,rank_by_revenue,rank_by_volume
0,health_beauty,8647,9465,"1,233,131.72",130.28,1,2
1,watches_gifts,5495,5859,"1,166,176.98",199.04,2,7
2,bed_bath_table,9272,10953,"1,023,434.76",93.44,3,1
3,sports_leisure,7530,8431,"954,852.55",113.25,4,3
4,computers_accessories,6530,7644,"888,724.61",116.26,5,5
5,furniture_decor,6307,8160,"711,927.69",87.25,6,4
6,housewares,5743,6795,"615,628.69",90.60,7,6
7,cool_stuff,3559,3718,"610,204.10",164.12,8,12
8,auto,3810,4140,"578,966.65",139.85,9,10
9,toys,3804,4030,"471,286.48",116.94,10,11


In [11]:
q8 = con.execute("""
SELECT 
    COALESCE(ct.product_category_name_english, p.product_category_name) AS category,
    COUNT(*) AS items_sold,
    ROUND(AVG(oi.price), 2) AS avg_price,
    ROUND(MEDIAN(oi.price), 2) AS median_price,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN category_translation ct 
    ON p.product_category_name = ct.product_category_name
JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
HAVING COUNT(*) >= 100
ORDER BY avg_price DESC
LIMIT 15
""").df()

print("=" * 60)
print("📊 Q8 — CATEGORIAS COM MAIOR TICKET MÉDIO")
print("=" * 60)
print(f"💎 Top 3 categorias mais caras (mín. 100 vendas):")
for _, row in q8.head(3).iterrows():
    print(f"   {row['category']}: R$ {row['avg_price']:,.2f} (mediana: R$ {row['median_price']:,.2f})")
print("=" * 60)

q8

📊 Q8 — CATEGORIAS COM MAIOR TICKET MÉDIO
💎 Top 3 categorias mais caras (mín. 100 vendas):
   computers: R$ 1,098.92 (mediana: R$ 1,100.00)
   home_appliances_2: R$ 467.33 (mediana: R$ 227.99)
   agro_industry_and_commerce: R$ 342.55 (mediana: R$ 243.75)


,category,items_sold,avg_price,median_price,total_revenue
0,computers,199,"1,098.92","1,100.00","218,684.14"
1,home_appliances_2,231,467.33,227.99,"107,953.95"
2,agro_industry_and_commerce,206,342.55,243.75,"70,566.10"
3,musical_instruments,651,283.13,94.90,"184,315.74"
4,small_appliances,658,277.74,98.36,"182,754.12"
5,fixed_telephony,255,216.92,49.00,"55,315.21"
6,construction_tools_safety,183,211.88,115.00,"38,773.22"
7,watches_gifts,5859,199.04,128.99,"1,166,176.98"
8,furniture_bedroom,103,184.97,179.00,"19,051.80"
9,air_conditioning,289,184.51,139.99,"53,323.56"


In [12]:
q9 = con.execute("""
SELECT 
    CASE 
        WHEN p.product_weight_g < 500 THEN '1 - Under 500g'
        WHEN p.product_weight_g < 2000 THEN '2 - 500g to 2kg'
        WHEN p.product_weight_g < 10000 THEN '3 - 2kg to 10kg'
        WHEN p.product_weight_g < 30000 THEN '4 - 10kg to 30kg'
        ELSE '5 - Over 30kg'
    END AS weight_range,
    COUNT(*) AS items,
    ROUND(AVG(oi.price), 2) AS avg_price,
    ROUND(AVG(oi.freight_value), 2) AS avg_freight,
    ROUND(100.0 * AVG(oi.freight_value) / AVG(oi.price), 2) AS freight_as_pct_of_price
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
WHERE p.product_weight_g IS NOT NULL
GROUP BY 1
ORDER BY 1
""").df()

print("=" * 60)
print("📊 Q9 — PESO DO PRODUTO VS FRETE")
print("=" * 60)
leve = q9.iloc[0]
pesado = q9.iloc[-1]
print(f"📦 Produtos {leve['weight_range'][4:]}: frete = {leve['freight_as_pct_of_price']}% do preço")
print(f"📦 Produtos {pesado['weight_range'][4:]}: frete = {pesado['freight_as_pct_of_price']}% do preço")
print(f"\n💡 INSIGHT: Produtos leves têm frete proporcionalmente {'MAIOR' if leve['freight_as_pct_of_price'] > pesado['freight_as_pct_of_price'] else 'MENOR'} ao preço")
print("=" * 60)

q9

📊 Q9 — PESO DO PRODUTO VS FRETE
📦 Produtos Under 500g: frete = 20.17% do preço
📦 Produtos Over 30kg: frete = 18.17% do preço

💡 INSIGHT: Produtos leves têm frete proporcionalmente MAIOR ao preço


,weight_range,items,avg_price,avg_freight,freight_as_pct_of_price
0,1 - Under 500g,44790,75.36,15.20,20.17
1,2 - 500g to 2kg,42104,112.52,17.78,15.80
2,3 - 2kg to 10kg,20413,181.11,26.06,14.39
3,4 - 10kg to 30kg,5040,316.00,50.89,16.10
4,5 - Over 30kg,285,654.85,118.96,18.17


In [13]:
q10 = con.execute("""
SELECT 
    c.customer_state,
    COUNT(*) AS delivered_orders,
    ROUND(AVG(DATE_DIFF('day', 
        o.order_purchase_timestamp, 
        o.order_delivered_customer_date)), 1) AS avg_delivery_days,
    ROUND(100.0 * SUM(CASE 
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date 
        THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_delivery_pct
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
    AND o.order_delivered_customer_date IS NOT NULL
GROUP BY 1
ORDER BY avg_delivery_days DESC
""").df()

print("=" * 60)
print("📊 Q10 — TEMPO DE ENTREGA POR ESTADO")
print("=" * 60)
pior = q10.iloc[0]
melhor = q10.iloc[-1]
print(f"🐢 Pior estado: {pior['customer_state']} — {pior['avg_delivery_days']} dias ({pior['late_delivery_pct']}% atrasados)")
print(f"⚡ Melhor estado: {melhor['customer_state']} — {melhor['avg_delivery_days']} dias ({melhor['late_delivery_pct']}% atrasados)")
print(f"\n📊 Diferença: {pior['avg_delivery_days'] - melhor['avg_delivery_days']:.1f} dias entre o pior e o melhor")
print("=" * 60)

q10

📊 Q10 — TEMPO DE ENTREGA POR ESTADO
🐢 Pior estado: RR — 29.3 dias (12.2% atrasados)
⚡ Melhor estado: SP — 8.7 dias (5.89% atrasados)

📊 Diferença: 20.6 dias entre o pior e o melhor


,customer_state,delivered_orders,avg_delivery_days,late_delivery_pct
0,RR,41,29.30,12.20
1,AP,67,27.20,4.48
2,AM,145,26.40,4.14
3,AL,397,24.50,23.93
4,PA,946,23.70,12.37
5,SE,335,21.50,15.22
6,MA,717,21.50,19.67
7,CE,1279,21.20,15.32
8,AC,80,21.00,3.75
9,PB,517,20.40,11.03


In [14]:

q11 = con.execute("""
SELECT 
    CASE 
        WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date 
        THEN 'On time'
        ELSE 'Late'
    END AS delivery_status,
    COUNT(*) AS total_orders,
    ROUND(AVG(CAST(r.review_score AS DOUBLE)), 2) AS avg_review_score,
    ROUND(100.0 * SUM(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END) 
          / COUNT(*), 2) AS pct_negative_reviews,
    ROUND(100.0 * SUM(CASE WHEN r.review_score = 5 THEN 1 ELSE 0 END) 
          / COUNT(*), 2) AS pct_five_stars
FROM orders o
INNER JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
    AND o.order_estimated_delivery_date IS NOT NULL
    AND r.review_score IS NOT NULL
GROUP BY 1
ORDER BY 1
""").df()

print("=" * 60)
print("⭐ Q11 — IMPACTO DE ATRASO NA NOTA (INSIGHT PRINCIPAL)")
print("=" * 60)

if len(q11) == 2:
    on_time = q11[q11['delivery_status'] == 'On time'].iloc[0]
    late = q11[q11['delivery_status'] == 'Late'].iloc[0]
    
    queda = on_time['avg_review_score'] - late['avg_review_score']
    queda_pct = 100 * queda / on_time['avg_review_score']
    total = on_time['total_orders'] + late['total_orders']
    pct_atraso = 100 * late['total_orders'] / total
    
    print(f"✅ No prazo: nota média {on_time['avg_review_score']}/5 ({int(on_time['total_orders']):,} pedidos)")
    print(f"❌ Atrasado: nota média {late['avg_review_score']}/5 ({int(late['total_orders']):,} pedidos)")
    print(f"\n🔥 QUEDA DE {queda:.2f} PONTOS ({queda_pct:.1f}% menor) quando atrasa")
    print(f"📊 {pct_atraso:.1f}% dos pedidos chegam atrasados")
    print(f"📉 % de reviews negativas: {on_time['pct_negative_reviews']}% (no prazo) vs {late['pct_negative_reviews']}% (atrasado)")
    print(f"\n💡 ESSE É O NÚMERO DO LINKEDIN POST!")
else:
    print("⚠️  Resultado inesperado. Mostrando dataframe pra debug:")

print("=" * 60)
q11

⭐ Q11 — IMPACTO DE ATRASO NA NOTA (INSIGHT PRINCIPAL)
✅ No prazo: nota média 4.29/5 (88,658 pedidos)
❌ Atrasado: nota média 2.57/5 (7,701 pedidos)

🔥 QUEDA DE 1.72 PONTOS (40.1% menor) quando atrasa
📊 8.0% dos pedidos chegam atrasados
📉 % de reviews negativas: 9.24% (no prazo) vs 54.02% (atrasado)

💡 ESSE É O NÚMERO DO LINKEDIN POST!


,delivery_status,total_orders,avg_review_score,pct_negative_reviews,pct_five_stars
0,Late,7701,2.57,54.02,22.22
1,On time,88658,4.29,9.24,62.43


In [15]:
q12 = con.execute("""
WITH orders_by_category AS (
    SELECT DISTINCT
        o.order_id,
        o.order_status,
        COALESCE(ct.product_category_name_english, p.product_category_name) AS category
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN category_translation ct 
        ON p.product_category_name = ct.product_category_name
)
SELECT 
    category,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
    ROUND(100.0 * SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) 
          / COUNT(*), 2) AS cancellation_rate_pct
FROM orders_by_category
WHERE category IS NOT NULL
GROUP BY 1
HAVING COUNT(*) >= 100
ORDER BY cancellation_rate_pct DESC
LIMIT 15
""").df()

print("=" * 60)
print("📊 Q12 — TAXA DE CANCELAMENTO POR CATEGORIA")
print("=" * 60)
print(f"🚫 Top 3 categorias com mais cancelamento:")
for _, row in q12.head(3).iterrows():
    print(f"   {row['category']}: {row['cancellation_rate_pct']}% ({int(row['canceled_orders'])} de {int(row['total_orders'])} pedidos)")
print(f"\n💡 INSIGHT: Investigar fornecedores dessas categorias pode reduzir churn")
print("=" * 60)

q12

📊 Q12 — TAXA DE CANCELAMENTO POR CATEGORIA
🚫 Top 3 categorias com mais cancelamento:
   fixed_telephony: 1.38% (3 de 217 pedidos)
   books_general_interest: 1.37% (7 de 512 pedidos)
   home_appliances_2: 1.28% (3 de 234 pedidos)

💡 INSIGHT: Investigar fornecedores dessas categorias pode reduzir churn


,category,total_orders,canceled_orders,cancellation_rate_pct
0,fixed_telephony,217,3.00,1.38
1,books_general_interest,512,7.00,1.37
2,home_appliances_2,234,3.00,1.28
3,musical_instruments,628,8.00,1.27
4,small_appliances,630,8.00,1.27
5,construction_tools_safety,167,2.00,1.20
6,costruction_tools_garden,194,2.00,1.03
7,fashion_male_clothing,112,1.00,0.89
8,food_drink,227,2.00,0.88
9,fashion_shoes,240,2.00,0.83


In [16]:
bonus = con.execute("""
SELECT 
    op.payment_type,
    COUNT(DISTINCT op.order_id) AS total_orders,
    ROUND(100.0 * COUNT(DISTINCT op.order_id) 
          / SUM(COUNT(DISTINCT op.order_id)) OVER (), 2) AS pct_of_orders,
    ROUND(AVG(op.payment_value), 2) AS avg_payment,
    ROUND(AVG(op.payment_installments), 2) AS avg_installments
FROM order_payments op
GROUP BY 1
ORDER BY total_orders DESC
""").df()

print("=" * 60)
print("📊 BÔNUS — MÉTODOS DE PAGAMENTO")
print("=" * 60)
top = bonus.iloc[0]
print(f"💳 Método mais usado: {top['payment_type']} ({top['pct_of_orders']}% dos pedidos)")
print(f"📊 Parcelamento médio (top): {top['avg_installments']:.1f}x")
print("=" * 60)

bonus

📊 BÔNUS — MÉTODOS DE PAGAMENTO
💳 Método mais usado: credit_card (75.24% dos pedidos)
📊 Parcelamento médio (top): 3.5x


,payment_type,total_orders,pct_of_orders,avg_payment,avg_installments
0,credit_card,76505,75.24,163.32,3.51
1,boleto,19784,19.46,145.03,1.00
2,voucher,3866,3.80,65.70,1.00
3,debit_card,1528,1.50,142.57,1.00
4,not_defined,3,0.00,0.00,1.00


In [2]:
import duckdb
import os


con = duckdb.connect('olist.db')


os.makedirs('data/processed', exist_ok=True)


tabelas = con.execute("SHOW TABLES").df()
print("📋 Tabelas no banco:")
print(tabelas)
print()

if 'fact_orders' not in tabelas['name'].values:
    print("⚠️  fact_orders não existe ainda. Criando...")
    con.execute("""
    CREATE OR REPLACE TABLE fact_orders AS
    SELECT 
        o.order_id,
        o.customer_id,
        c.customer_unique_id,
        c.customer_city,
        c.customer_state,
        o.order_status,
        o.order_purchase_timestamp,
        DATE_TRUNC('month', o.order_purchase_timestamp) AS order_month,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        DATE_DIFF('day', o.order_purchase_timestamp, 
                  o.order_delivered_customer_date) AS delivery_days,
        CASE 
            WHEN o.order_delivered_customer_date <= o.order_estimated_delivery_date 
            THEN 'On time' ELSE 'Late' 
        END AS delivery_status,
        oi.product_id,
        COALESCE(ct.product_category_name_english, p.product_category_name) AS category,
        p.product_weight_g,
        oi.seller_id,
        oi.price,
        oi.freight_value,
        oi.price + oi.freight_value AS total_value,
        r.review_score
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    LEFT JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN category_translation ct ON p.product_category_name = ct.product_category_name
    LEFT JOIN order_reviews r ON o.order_id = r.order_id
    """)
    print("✅ Tabela criada!")


con.execute("""
COPY fact_orders TO 'data/processed/fact_orders.csv' (HEADER, DELIMITER ',')
""")


caminho_completo = os.path.abspath('data/processed/fact_orders.csv')
tamanho_mb = os.path.getsize(caminho_completo) / (1024 * 1024)
linhas = con.execute("SELECT COUNT(*) FROM fact_orders").fetchone()[0]

print()
print("=" * 60)
print("✅ SUCESSO!")
print("=" * 60)
print(f"📁 Caminho COMPLETO do arquivo:")
print(f"   {caminho_completo}")
print(f"📦 Tamanho: {tamanho_mb:.1f} MB")
print(f"📊 Linhas: {linhas:,}")
print("=" * 60)
print()

📋 Tabelas no banco:
                   name
0  category_translation
1             customers
2           fact_orders
3           order_items
4        order_payments
5         order_reviews
6                orders
7              products
8               sellers


✅ SUCESSO!
📁 Caminho COMPLETO do arquivo:
   C:\Users\Zyon\OneDrive\Área de Trabalho\olist-analysis\data\processed\fact_orders.csv
📦 Tamanho: 34.5 MB
📊 Linhas: 113,314



In [4]:
import duckdb
import pandas as pd
import os

con = duckdb.connect('olist.db')
os.makedirs('data/processed', exist_ok=True)


df = con.execute("SELECT * FROM fact_orders").df()


df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['freight_value'] = pd.to_numeric(df['freight_value'], errors='coerce')
df['total_value'] = pd.to_numeric(df['total_value'], errors='coerce')
df['product_weight_g'] = pd.to_numeric(df['product_weight_g'], errors='coerce')
df['review_score'] = pd.to_numeric(df['review_score'], errors='coerce')
df['delivery_days'] = pd.to_numeric(df['delivery_days'], errors='coerce')


df.to_csv(
    'data/processed/fact_orders.csv',
    index=False,
    sep=',',          # vírgula separa colunas
    decimal='.',      # ponto separa decimais
    encoding='utf-8'
)

print(f"✅ Arquivo salvo!")
print(f"💰 Receita total: R$ {df['total_value'].sum():,.2f}")
print(f"📊 Linhas: {len(df):,}")
print(f"📋 Pedidos únicos: {df['order_id'].nunique():,}")
print(f"\n🔍 Amostra:")
print(df[['order_id', 'price', 'freight_value', 'total_value']].head())

✅ Arquivo salvo!
💰 Receita total: R$ 15,915,872.32
📊 Linhas: 113,314
📋 Pedidos únicos: 98,666

🔍 Amostra:
                           order_id   price  freight_value  total_value
0  00010242fe8c5a6d1ba2dd792cb16214   58.90          13.29        72.19
1  00018f77f2f0320c557190d7a144bdd3  239.90          19.93       259.83
2  000229ec398224ef6ca0657da4fc703e  199.00          17.87       216.87
3  00024acbcdf0a6daa1e931b038114c75   12.99          12.79        25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9  199.90          18.14       218.04
